In [18]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "assignment2" / "data" / "db"
TEST_INPUT_PATH = (
    PROJECT_ROOT
    / "assignment2"
    / "submission"
    / "test_input.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "outputs"

sys.path.insert(0, str(PROJECT_ROOT))

from src.features import (
    build_activity_events,
    build_features,
)

In [19]:
users = pd.read_parquet(
    DATA_DIR / "Users.parquet"
)

In [20]:
test_users = test_input.merge(
    users[
        [
            "user_id",
            "github_user_id"
        ]
    ],
    left_on="UserID",
    right_on="github_user_id",
    how="left"
)

test_users.head()

,UserID,user_id,github_user_id
0,100431196554039,10069,100431196554039
1,100457905686263,35425,100457905686263
2,101051236896852,14931,101051236896852
3,101306412508088,20331,101306412508088
4,101380187937965,11261,101380187937965


In [21]:
FINAL_PREDICTION_TIME = pd.Timestamp(
    "2026-11-22 00:00:00",
    tz="Asia/Taipei"
).tz_convert("UTC")

FINAL_PREDICTION_TIME

Timestamp('2026-11-21 16:00:00+0000', tz='UTC')

In [22]:
prs = pd.read_parquet(DATA_DIR / "PullRequests.parquet")
issues = pd.read_parquet(DATA_DIR / "Issues.parquet")
issue_comments = pd.read_parquet(DATA_DIR / "IssueComments.parquet")
pr_reviews = pd.read_parquet(DATA_DIR / "PRReviews.parquet")
pr_review_comments = pd.read_parquet(DATA_DIR / "PRReviewComments.parquet")
pushes = pd.read_parquet(DATA_DIR / "Pushes.parquet")

activity_events = build_activity_events(
    prs=prs,
    issues=issues,
    issue_comments=issue_comments,
    pr_reviews=pr_reviews,
    pr_review_comments=pr_review_comments,
    pushes=pushes
)

In [23]:
all_final_features = build_features(
    users=users,
    activity_events=activity_events,
    prediction_time=FINAL_PREDICTION_TIME,
    windows=[7, 30, 90]
)

print(all_final_features.shape)
all_final_features.head()

(9000, 36)


,user_id,github_user_id,login,open_pr_7d,open_issue_7d,issue_comment_7d,pr_review_7d,pr_review_comment_7d,push_7d,distinct_repos_7d,...,active_days_90d,days_since_last_open_pr,days_since_last_open_issue,days_since_last_issue_comment,days_since_last_pr_review,days_since_last_pr_review_comment,days_since_last_push,total_historical_events,total_historical_repos,prediction_time
0,0,912728859988373,aaaba9dd7a4f5396,0,0,0,0,0,0,0,...,10,9999.000000,167.205648,167.180822,9999.000000,9999.000000,26.659005,395,1,2026-11-21 16:00:00+00:00
1,1,883531231651856,ad70593dbeff5b03,0,0,0,0,0,0,0,...,45,181.417697,9999.000000,99.942801,9999.000000,9999.000000,15.342338,4041,1,2026-11-21 16:00:00+00:00
2,2,293581896069444,cfd134fe30c776d4,0,0,0,0,0,0,0,...,5,30.664873,218.193056,217.315162,9999.000000,9999.000000,31.047674,93,11,2026-11-21 16:00:00+00:00
3,3,580828034442701,a091ff9c6adb3489,0,0,0,0,0,0,0,...,13,64.262407,67.196979,66.959653,14.274248,53.620775,49.342604,501,44,2026-11-21 16:00:00+00:00
4,4,880766511796264,0bab9b4398d5271c,0,0,0,0,0,0,0,...,56,37.510521,13.700127,14.978368,15.577442,27.747593,26.971875,2242,50,2026-11-21 16:00:00+00:00


In [24]:
test_features = test_users[
    ["UserID", "user_id"]
].merge(
    all_final_features,
    on="user_id",
    how="left"
)

print("Number of test users:", len(test_features))

test_features.head()

Number of test users: 4465


,UserID,user_id,github_user_id,login,open_pr_7d,open_issue_7d,issue_comment_7d,pr_review_7d,pr_review_comment_7d,push_7d,...,active_days_90d,days_since_last_open_pr,days_since_last_open_issue,days_since_last_issue_comment,days_since_last_pr_review,days_since_last_pr_review_comment,days_since_last_push,total_historical_events,total_historical_repos,prediction_time
0,100431196554039,10069,100431196554039,dab81ea8118aa354,0,0,0,0,0,0,...,53,14.675139,14.602500,12.377014,9999.0,9999.000000,27.534167,796,11,2026-11-21 16:00:00+00:00
1,100457905686263,35425,100457905686263,4cb0e9f1c7914351,0,0,0,0,0,0,...,0,188.782141,9999.000000,9999.000000,9999.0,9999.000000,93.194745,39,7,2026-11-21 16:00:00+00:00
2,101051236896852,14931,101051236896852,a688921b3e438c59,0,0,0,0,0,0,...,3,187.765949,175.836076,9999.000000,9999.0,9999.000000,27.034051,246,8,2026-11-21 16:00:00+00:00
3,101306412508088,20331,101306412508088,1d9c6374da667b5f,0,0,0,0,0,0,...,2,9999.000000,67.336400,9999.000000,9999.0,9999.000000,89.262546,57,13,2026-11-21 16:00:00+00:00
4,101380187937965,11261,101380187937965,8f456dec76f56f02,0,0,0,0,0,0,...,33,23.159051,23.201389,23.408924,9999.0,173.109965,27.432188,2479,99,2026-11-21 16:00:00+00:00


In [25]:
import numpy as np

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight


ACTIVITIES = [
    "open_pr",
    "open_issue",
    "issue_comment",
    "pr_review",
    "pr_review_comment",
    "push",
]

LABEL_COLUMNS = [
    f"label_{activity}"
    for activity in ACTIVITIES
]


training_df = pd.read_parquet(
    OUTPUT_DIR / "training_features.parquet"
)

threshold_df = pd.read_csv(
    OUTPUT_DIR / "selected_hgb_thresholds.csv"
)

selected_thresholds = dict(
    zip(
        threshold_df["activity"],
        threshold_df["threshold"],
    )
)

print(selected_thresholds)

{'open_pr': 0.75, 'open_issue': 0.9, 'issue_comment': 0.9, 'pr_review': 0.8500000000000001, 'pr_review_comment': 0.9, 'push': 0.5}


In [26]:
EXCLUDE_COLUMNS = [
    "user_id",
    "github_user_id",
    "login",
    "prediction_time",
] + LABEL_COLUMNS

FEATURE_COLUMNS = [
    column
    for column in training_df.columns
    if column not in EXCLUDE_COLUMNS
]

print("Number of features:", len(FEATURE_COLUMNS))

missing_features = [
    column
    for column in FEATURE_COLUMNS
    if column not in test_features.columns
]

if missing_features:
    raise ValueError(
        f"Test data is missing features: {missing_features}"
    )

Number of features: 32


In [27]:
X_train = training_df[
    FEATURE_COLUMNS
].copy()

X_test = test_features[
    FEATURE_COLUMNS
].copy()

if X_train.isna().any().any():
    raise ValueError(
        "Training features contain missing values."
    )

if X_test.isna().any().any():
    missing_columns = (
        X_test.columns[
            X_test.isna().any()
        ]
        .tolist()
    )

    raise ValueError(
        "Test features contain missing values in: "
        f"{missing_columns}"
    )

print("Training matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

Training matrix: (315000, 32)
Test matrix: (4465, 32)


In [28]:
final_models = {}
final_probabilities = {}
final_predictions = pd.DataFrame(
    index=test_features.index
)

for activity in ACTIVITIES:
    label = f"label_{activity}"

    y_train = (
        training_df[label]
        .astype(int)
    )

    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=42,
    )

    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weights,
    )

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    threshold = float(
        selected_thresholds[activity]
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    final_models[activity] = model
    final_probabilities[activity] = probabilities
    final_predictions[activity] = predictions

    print(
        f"{activity:<20} "
        f"threshold={threshold:.2f} "
        f"positives={predictions.sum()}"
    )

open_pr              threshold=0.75 positives=32
open_issue           threshold=0.90 positives=34
issue_comment        threshold=0.90 positives=13
pr_review            threshold=0.85 positives=66
pr_review_comment    threshold=0.90 positives=27
push                 threshold=0.50 positives=197


In [29]:
submission = pd.DataFrame({
    "UserID": test_features["UserID"].values
})

for activity in ACTIVITIES:
    submission[activity] = (
        final_predictions[activity].values
    )

submission = submission[
    [
        "UserID",
        "open_pr",
        "open_issue",
        "issue_comment",
        "pr_review",
        "pr_review_comment",
        "push",
    ]
]

submission.head()

,UserID,open_pr,open_issue,issue_comment,pr_review,pr_review_comment,push
0,100431196554039,0,1,0,0,0,0
1,100457905686263,0,0,0,0,0,0
2,101051236896852,0,0,0,0,0,0
3,101306412508088,0,0,0,0,0,0
4,101380187937965,0,0,0,0,0,0


In [32]:
STUDENT_ID = "115550819"

OUTPUT_PATH = (
    OUTPUT_DIR
    / f"output_{STUDENT_ID}.csv"
)

submission.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved:", OUTPUT_PATH)

Saved: /Users/triciachew/BigData2026/outputs/output_115550819.csv
